# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use Logistic Regression because the task is to identify content items that are likely to be declining and rank them for decision support.

Logistic Regression is a simple and interpretable classification method. It produces a probability score that can be used to rank content items, which fits the ranked-queue goal of the Week-4 baseline.

I chose it before trying more complex models because the first model should be easy to inspect and compare fairly against the hand-written baseline.

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Method: Logistic Regression")
print("Task: binary classification of declining vs non-declining content")
print("Ranking metric: precision@50")

Method: Logistic Regression
Task: binary classification of declining vs non-declining content
Ranking metric: precision@50


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped train/test split by client_id, with 80% of clients for training and 20% held out for testing.

This is more honest than randomly splitting rows because pages from the same client can share similar characteristics. Holding clients out tests whether the model generalises to unseen clients.

The split is fixed with a random seed so the comparison can be reproduced.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# Load the same 30,000-row starter dataset used by the Week-4 baseline
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / "data").exists():
        break
    repo_root = repo_root.parent

DATA_PATH = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Clients:", df["client_id"].nunique())

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())
print(
    "Client overlap:",
    len(set(train_df["client_id"]) & set(test_df["client_id"]))
)

Dataset shape: (30000, 44)
Clients: 32
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

The target is the observed declining label: 1 when trend_direction is "down", otherwise 0.

I exclude trend_direction and trend_pct because they define the target and would leak the answer into the features. I also exclude the last-30-day impression columns because they directly contribute to the trend label.

The model and Week-4 baseline are evaluated on the same held-out clients using precision@50.

The Week-4 baseline is reproduced using its original transparent rule: staleness, search demand, ranking position, and CTR.

In [4]:
from sklearn.metrics import precision_score

# ---------------------------------------------------------
# 1. Create the observed target for evaluation/training
# ---------------------------------------------------------
train_df["target"] = (
    train_df["trend_direction"] == "down"
).astype(int)

test_df["target"] = (
    test_df["trend_direction"] == "down"
).astype(int)

print("Train decline rate:", round(train_df["target"].mean(), 3))
print("Test decline rate:", round(test_df["target"].mean(), 3))


# ---------------------------------------------------------
# 2. Features
# ---------------------------------------------------------
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
]

excluded = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
}

numeric_features = [
    c for c in numeric_features
    if c not in excluded
]

categorical_features = [
    c for c in categorical_features
    if c not in excluded
]


# ---------------------------------------------------------
# 3. Preprocessing + Logistic Regression
# ---------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", __import__("sklearn").impute.SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", __import__("sklearn").impute.SimpleImputer(
                    strategy="most_frequent"
                )),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        )
    )
])

X_train = train_df[numeric_features + categorical_features]
y_train = train_df["target"]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df["target"]

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 4. Week-4 baseline
# ---------------------------------------------------------
baseline = pd.DataFrame(index=test_df.index)

baseline["score"] = (
    (
        (test_df["days_since_last_update"].fillna(0) > 180).astype(int) * 40
        + (test_df["search_volume"].fillna(0) >= 100).astype(int) * 30
        + (
            test_df["avg_position"].fillna(0).between(8, 20)
        ).astype(int) * 20
        + (test_df["ctr"].fillna(0) < 2).astype(int) * 10
    )
)

baseline_scores = baseline["score"].to_numpy()


# ---------------------------------------------------------
# 5. Precision@50 helper
# ---------------------------------------------------------
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_k = np.argsort(scores)[::-1][:k]

    return y_true[top_k].mean()


baseline_p50 = precision_at_k(
    y_test.to_numpy(),
    baseline_scores,
    k=50
)

model_p50 = precision_at_k(
    y_test.to_numpy(),
    model_scores,
    k=50
)

base_rate = y_test.mean()


# ---------------------------------------------------------
# 6. Comparison table
# ---------------------------------------------------------
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ],
    "test_base_rate": [
        base_rate,
        base_rate
    ]
})

comparison["precision_at_50"] = (
    comparison["precision_at_50"].round(3)
)

comparison["test_base_rate"] = (
    comparison["test_base_rate"].round(3)
)

display(comparison)

Train decline rate: 0.55
Test decline rate: 0.511


,method,precision_at_50,test_base_rate
0,Week-4 baseline,0.54,0.511
1,Logistic Regression,0.72,0.511


The Logistic Regression model achieved a Precision@50 of 0.72 on the test set, compared with 0.54 for the Week-4 baseline. This is an 18 percentage-point improvement on the same client-grouped test split. The test decline rate was 0.511 for both methods. The model therefore provides stronger decision-support than the baseline on this metric, although the result should not be treated as proof that the model will generalize to future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Logistic Regression model produced 14 false positives in its top-50 predictions. These are cases where the model assigned a high score and selected the content for the top-50, but the observed test target was 0. This shows that the model can over-rank some content even when the measured outcome does not support the selection.

The largest absolute coefficients were for users and sessions over the 90-day historical window, followed by days with impressions and days with sessions. This suggests the model is leaning mainly on historical traffic and activity signals. Content age and content type also contributed to the ranking.

The model achieved a measured Precision@50 of 0.720 compared with 0.540 for the Week-4 baseline on the same client-grouped test set. This is an 18 percentage-point improvement in this test. The result is useful as decision-support, but the false positives show that a high model score is not always correct. In particular, low-traffic or unusual content can still be ranked highly without the expected outcome.

The interpretation is directional rather than causal: the coefficients show which features the model uses for prediction, not that those features cause content decline.

In [5]:
# ---------------------------------------------------------
# 1. Error analysis
# ---------------------------------------------------------
error_df = test_df[
    [
        "content_id",
        "client_id",
        "target",
        "search_volume",
        "days_since_last_update",
        "avg_position",
        "ctr"
    ]
].copy()

error_df["model_score"] = model_scores
error_df["predicted_top50"] = 0

top50_idx = np.argsort(model_scores)[::-1][:50]

error_df.iloc[top50_idx, error_df.columns.get_loc("predicted_top50")] = 1

false_positives = error_df[
    (error_df["predicted_top50"] == 1)
    & (error_df["target"] == 0)
].sort_values(
    "model_score",
    ascending=False
)

print("False positives in model top-50:", len(false_positives))

display(false_positives.head(3))


# ---------------------------------------------------------
# 2. Model coefficients
# ---------------------------------------------------------
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": np.abs(coefficients)
}).sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top model features by absolute coefficient:")

display(
    importance[
        ["feature", "coefficient"]
    ].head(10)
)


# ---------------------------------------------------------
# 3. Short interpretation
# ---------------------------------------------------------
print(
    f"Observed test precision@50 was "
    f"{model_p50:.3f} for Logistic Regression versus "
    f"{baseline_p50:.3f} for the Week-4 baseline."
)

if model_p50 > baseline_p50:
    print(
        "The model ranked more declining pages in its top 50 "
        "than the baseline on this held-out client split."
    )
elif model_p50 < baseline_p50:
    print(
        "The baseline ranked more declining pages in its top 50 "
        "than the model on this held-out client split."
    )
else:
    print(
        "The model and baseline had the same precision@50 "
        "on this held-out client split."
    )

print(
    "The coefficient table gives directional associations, "
    "not causal effects."
)

False positives in model top-50: 14


,content_id,client_id,target,search_volume,days_since_last_update,avg_position,ctr,model_score,predicted_top50
8016,content_c94a53e3bfb8,client_f369cb89fc,0,0.0,20,8.1,0.23,0.892796,1
10175,content_374e795aab68,client_f369cb89fc,0,880.0,20,31.0,0.85,0.892260,1
26614,content_7be5f150dc65,client_f369cb89fc,0,10.0,20,5.9,0.00,0.887934,1


Top model features by absolute coefficient:


,feature,coefficient
9,num__users_90d,-0.988487
8,num__sessions_90d,0.769966
13,num__days_with_impressions,0.603503
31,cat__main_intent_navigational,-0.450080
14,num__days_with_sessions,-0.426135
28,cat__content_type_keyword article,0.356660
27,cat__content_type_feedly article,-0.339396
15,num__content_age_days,-0.288695
12,num__scroll_events_90d,0.280714
3,num__word_count,0.242005


Observed test precision@50 was 0.720 for Logistic Regression versus 0.540 for the Week-4 baseline.
The model ranked more declining pages in its top 50 than the baseline on this held-out client split.
The coefficient table gives directional associations, not causal effects.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.